# PMPDB Multimodal Pain-Intensity Estimation — Exploration & Verification Notebook

This reproducible notebook verifies the local **PMPDB (PainMonit Physiotherapy Database)**, inspects processed features, and summarizes participant-independent model evaluation and explainability results.

> **Note**: PMPDB contains **6 peripheral modalities (8 channels @ 250 Hz)**: `BVP`, `EDA`, `Resp`, `Tmp`, `EMG`, `Grip`. Neither **ECG** nor **EEG** is recorded.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path('..').resolve()
summary = json.loads((ROOT / 'dataset/metadata/dataset_summary.json').read_text())
print('Dataset:', summary['dataset_name'])
print('Participants:', summary['num_participants'], '| Main Recordings:', summary['num_main_recordings'])
print('Sampling Rate:', summary['sampling_frequency_hz'], 'Hz | Total Hours:', summary['duration_stats_sec']['total_hours'])
print('Available Modalities:', list(summary['available_modalities'].keys()))
print('Absent Modalities:', list(summary['absent_modalities'].keys()))

In [ ]:
feat_df = pd.read_parquet(ROOT / 'dataset/processed/features.parquet')
print('Processed feature table shape:', feat_df.shape)
feat_df[['recording_id', 'participant_id', 'timestamp_sec', 'pain_score_0_100', 'pain_label', 'EMG_rms_ratio', 'EDA_RB_scl_mean', 'BVP_E4_pr_bpm', 'Grip_rel_runup_z']].head(10)

In [ ]:
base = json.loads((ROOT / 'experiments/baseline_results.json').read_text())
deep = json.loads((ROOT / 'experiments/deep_results.json').read_text())
hyb = json.loads((ROOT / 'experiments/hybrid_results.json').read_text())
rows = []
for k, v in base['groupkfold'].items():
    rows.append({'Model': k, **{m: v[m] for m in ['mae', 'rmse', 'pearson_r', 'ccc', 'r2']}})
for k, v in deep['models'].items():
    rows.append({'Model': k, **{m: v[m] for m in ['mae', 'rmse', 'pearson_r', 'ccc', 'r2']}})
rows.append({'Model': 'Hybrid_MM_Gated_XGB', **{m: hyb['metrics'][m] for m in ['mae', 'rmse', 'pearson_r', 'ccc', 'r2']}})
pd.DataFrame(rows).sort_values('mae')